In [ ]:
import pandas as pd
import os
from sklearn.feature_selection import VarianceThreshold
import numpy as np
from sklearn.feature_selection import SelectKBest, f_classif
from mrmr import mrmr_classif
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer


In [2]:
print(os.getcwd())
os.chdir('C:\\Users\\moham\\Downloads\\Projects\\Toxicity_Prediction\\Datasets')
os.listdir()

c:\Users\moham\Downloads\Projects\Toxicity_Prediction\Preprocessing


['ml_data_0.9.xlsx', 'radiomic_features.parquet']

In [3]:
clinical_data = pd.read_excel('ml_data_0.9.xlsx')
radiomic_data = pd.read_parquet('radiomic_features.parquet')

In [4]:
radiomic_data

,Oesophagus_original_firstorder_Energy,Oesophagus_original_firstorder_TotalEnergy,Oesophagus_original_firstorder_Entropy,Oesophagus_original_firstorder_Minimum,Oesophagus_original_firstorder_10Percentile,Oesophagus_original_firstorder_90Percentile,Oesophagus_original_firstorder_Maximum,Oesophagus_original_firstorder_Mean,Oesophagus_original_firstorder_Median,Oesophagus_original_firstorder_InterquartileRange,...,PTV HR_gradient_gldm_DependenceNonUniformity,PTV HR_gradient_gldm_DependenceNonUniformityNormalized,PTV HR_gradient_gldm_DependenceVariance,PTV HR_gradient_gldm_DependenceEntropy,PTV HR_gradient_gldm_LowGrayLevelEmphasis,PTV HR_gradient_gldm_HighGrayLevelEmphasis,PTV HR_gradient_gldm_SmallDependenceLowGrayLevelEmphasis,PTV HR_gradient_gldm_SmallDependenceHighGrayLevelEmphasis,PTV HR_gradient_gldm_LargeDependenceLowGrayLevelEmphasis,PTV HR_gradient_gldm_LargeDependenceHighGrayLevelEmphasis
Patient,,,,,,,,,,,,,,,,,,,,,
001-002,6.794106e+06,6.794106e+06,2.167484,-44.727386,5.316829,72.650925,169.369339,39.132837,38.957911,36.335053,...,6859.700061,0.048595,69.303638,6.112580,0.610995,16.370149,0.005649,3.746134,259.428307,654.634569
001-003,1.102086e+07,1.102086e+07,2.614822,-225.057159,-1.013463,91.812785,244.490646,49.727517,53.724943,46.364288,...,8859.442646,0.058526,69.860937,5.960787,0.626914,14.455905,0.005202,3.229413,280.701229,690.287859
001-004,7.831940e+06,7.831940e+06,2.523830,-313.172913,-0.390229,89.460600,125.497162,48.982656,57.596956,46.255467,...,6924.606134,0.051303,69.152033,6.170141,0.594033,18.292765,0.005298,3.878380,252.038925,732.186138
001-005,6.961838e+06,6.961838e+06,2.474564,-260.080688,-24.893004,66.035053,117.035904,30.011853,46.112614,43.010863,...,68644.398325,0.207903,60.112678,4.312041,0.805702,8.675139,0.003300,2.148789,457.520798,692.776037
001-006,9.653820e+06,9.653820e+06,2.254564,-323.972382,13.536752,82.292016,126.869629,47.159972,47.813440,37.936217,...,64001.582198,0.173303,67.692615,4.639645,0.779226,9.078694,0.003659,1.864231,435.181894,679.717582
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
064-003,7.063033e+06,7.063033e+06,1.999700,-123.861610,9.211510,67.048528,159.160904,41.091206,46.623339,30.020723,...,14130.143226,0.081823,78.702820,5.741060,0.661169,32.401217,0.004808,14.407075,326.413625,751.598412
064-004,3.348447e+07,3.348447e+07,2.591983,-932.109985,-2.680482,70.517316,433.541199,23.556834,45.240154,25.801770,...,13076.144841,0.076666,95.811134,6.032838,0.608456,47.324672,0.005368,16.974258,309.475779,826.804104
064-005,3.515137e+07,3.515137e+07,2.237662,-852.335876,12.951450,71.184705,146.968216,31.762665,48.494780,25.670817,...,55907.419230,0.162785,77.614016,4.818877,0.760487,17.488543,0.004005,5.546297,424.119485,699.866182


In [5]:
radiomic_data['Patient'] = clinical_data['REFERENCE']

### Variance Based Feature reduction in Radiomic Features

In [6]:
X_rad = radiomic_data.copy()

vt = VarianceThreshold(threshold=0.2)
X_rad = pd.DataFrame(
    vt.fit_transform(X_rad),
    columns=radiomic_data.columns[vt.get_support()]
)

print(X_rad.shape)

(607, 6662)


### Extracion of Highly Correlated Features

In [7]:
corr_matrix = X_rad.corr().abs()

upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

to_drop = [col for col in upper.columns if any(upper[col] > 0.80)]

X_rad = X_rad.drop(columns=to_drop)

print(X_rad.shape)

(607, 870)


### Clinical Feature Preprocessing

In [8]:
cols_to_drop = [
    'Unnamed: 0',
    'REFERENCE',
    '(MHYN) Medical Oncological Significant History',
    '(IMRTHPTV) Total Dose Received in Gy',
    'chemo_fit',
    'avelumab_given',
    'PFS_event',
    'PFS_duration_m',
    'Dysphagia_binary',
    'Dermatitis_binary',
    'Oralpain_binary',
    'Esophagus_Dnear-max',
    'Mandible_Dnear-max',
    'Oral_Cavity_Dnear-max'
]

clinical_data = clinical_data.drop(columns=cols_to_drop, errors='ignore')

In [9]:
x_clinical = pd.read_excel('ml_data_0.9.xlsx')
clinical_data['Patient'] = x_clinical['REFERENCE']

In [10]:
X_radiomics = X_rad.copy()
X_radiomics['Patient'] = radiomic_data.index.values

cols = ['Patient'] + [c for c in X_radiomics.columns if c != 'Patient']
X_radiomics = X_radiomics[cols]

print(X_radiomics.shape)

(607, 871)


In [11]:
print(X_radiomics.select_dtypes(exclude='number').columns.tolist())  # must be []  (no string cols)
print('feature NaNs:', X_radiomics.isna().sum().sum())               # must be 0

['Patient', 'Oesophagus_original_firstorder_Energy', 'Oesophagus_original_firstorder_Entropy', 'Oesophagus_original_firstorder_Minimum', 'Oesophagus_original_firstorder_10Percentile', 'Oesophagus_original_firstorder_90Percentile', 'Oesophagus_original_firstorder_Maximum', 'Oesophagus_original_firstorder_Median', 'Oesophagus_original_firstorder_InterquartileRange', 'Oesophagus_original_firstorder_Range', 'Oesophagus_original_firstorder_Skewness', 'Oesophagus_original_firstorder_Kurtosis', 'Oesophagus_original_shape_MeshVolume', 'Oesophagus_original_shape_Maximum2DDiameterSlice', 'Oesophagus_original_shape_Maximum2DDiameterColumn', 'Oesophagus_original_shape_LeastAxisLength', 'Oesophagus_original_glcm_ClusterShade', 'Oesophagus_original_glszm_LargeAreaEmphasis', 'Oesophagus_original_glszm_GrayLevelNonUniformity', 'Oesophagus_original_glszm_LargeAreaLowGrayLevelEmphasis', 'Oesophagus_original_glszm_LargeAreaHighGrayLevelEmphasis', 'Oesophagus_original_glrlm_LongRunEmphasis', 'Oesophagus_o

In [12]:
col = 'Oesophagus_original_firstorder_Energy'
coerced = pd.to_numeric(X_radiomics[col], errors='coerce')
bad = X_radiomics[col][coerced.isna() & X_radiomics[col].notna()]   # values that fail to convert
print(bad.unique())

[]


In [13]:
X_radiomics = X_radiomics.apply(pd.to_numeric, errors='coerce')

In [14]:
missing_fraction = X_radiomics.isnull().mean()

threshold = 0.05
X_radiomics = X_radiomics.loc[:, missing_fraction <= threshold]

print(f"Original features: {X_radiomics.shape[1]}")
print(f"Remaining features: {X_radiomics.shape[1]}")
print(f"Removed features: {X_radiomics.shape[1] - X_radiomics.shape[1]}")

Original features: 826
Remaining features: 826
Removed features: 0


In [15]:
X_radiomics['Patient'] = radiomic_data.index

In [16]:
X_combined = X_radiomics.merge(
    clinical_data,
    on="Patient",
    how="inner"
)

In [ ]:
y_x = X_combined["Drymouth_binary"].copy()

X_combined = X_combined.drop(
    columns=["Drymouth_binary", "Mucositis_binary"]
)

X_combined = X_combined.set_index("Patient")

X_features_x = X_combined.copy()

print("X_features_x:", X_features_x.shape)
print("y_x:", y_x.shape)

X_features_x: (594, 845)
X_features_m: (594, 845)
y_x: (594,)
y_m: (594,)


In [ ]:
# Xerostomia
X_train_x, X_test_x, y_train_x, y_test_x = train_test_split(
    X_features_x,
    y_x,
    test_size=0.2,
    stratify=y_x,
    random_state=42
)

imputer_x = SimpleImputer(strategy="median")

X_train_x = pd.DataFrame(
    imputer_x.fit_transform(X_train_x),
    columns=X_train_x.columns,
    index=X_train_x.index
)

X_test_x = pd.DataFrame(
    imputer_x.transform(X_test_x),
    columns=X_test_x.columns,
    index=X_test_x.index
)

# Check
print("X_train_x:", X_train_x.shape)
print("X_test_x:", X_test_x.shape)


X_train_x: (475, 845)
X_test_x: (119, 845)
X_train_m: (475, 845)
X_test_m: (119, 845)


In [ ]:

clinical_cols = [
    c for c in clinical_data.columns
    if c not in ["Patient", "Drymouth_binary", "Mucositis_binary"]
]

radiomics_cols = [
    c for c in X_train_x.columns
    if c not in clinical_cols
]

# Xerostomia
X_train_clinical_x = X_train_x[clinical_cols].copy()
X_test_clinical_x  = X_test_x[clinical_cols].copy()

X_train_rad_x = X_train_x[radiomics_cols].copy()
X_test_rad_x  = X_test_x[radiomics_cols].copy()


In [ ]:
print("X_train_clinical_x:", X_train_clinical_x.shape)
print("X_train_rad_x:", X_train_rad_x.shape)


X_train_clinical_x: (475, 19)
X_train_rad_x: (475, 826)
X_train_clinical_m: (475, 19)
X_train_rad_m: (475, 826)


## Split and MRMR Selction (Supervised Reduction)

In [ ]:
y_train_x.index = X_train_x.index
y_test_x.index = X_test_x.index


In [ ]:
K = 150

# Xerostomia
selected_rad_x_mrmr = mrmr_classif(
    X=X_train_rad_x,
    y=y_train_x,
    K=K
)

X_train_rad_x_mrmr = X_train_rad_x[selected_rad_x_mrmr].copy()


100%|██████████| 150/150 [00:14<00:00, 10.21it/s]


In [23]:
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
import pandas as pd

results_x = []

for feature in X_train_rad_x_mrmr.columns:

    group0 = X_train_rad_x_mrmr.loc[y_train_x == 0, feature]
    group1 = X_train_rad_x_mrmr.loc[y_train_x == 1, feature]

    _, p = mannwhitneyu(
        group0,
        group1,
        alternative="two-sided"
    )

    results_x.append([feature, p])

mw_x = pd.DataFrame(
    results_x,
    columns=["Feature", "p_value"]
)

# Benjamini-Hochberg correction
mw_x["p_adj"] = multipletests(
    mw_x["p_value"],
    method="fdr_bh"
)[1]

mw_x = mw_x.sort_values("p_adj")

mw_x.head()

,Feature,p_value,p_adj
26,Left parotid_wavelet-HHL_gldm_DependenceVariance,0.000011,0.001599
0,Mandible_wavelet-HHL_firstorder_Mean,0.000077,0.002243
24,PCM_original_firstorder_90Percentile,0.000035,0.002243
10,Oesophagus_wavelet-LLH_firstorder_Mean,0.000072,0.002243
44,PCM_wavelet-LHL_firstorder_90Percentile,0.000090,0.002243


In [24]:
sig_features_x = mw_x.loc[
    mw_x["p_adj"] < 0.05,
    "Feature"
].tolist()

X_train_rad_x_mrmr_mw = X_train_rad_x_mrmr[sig_features_x].copy()

print("Significant features:", len(sig_features_x))

Significant features: 110


In [27]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
import pandas as pd
import numpy as np

X = X_train_rad_x_mrmr_mw
y = y_train_x

# Initial RF to rank features
rf = RandomForestClassifier(
    n_estimators=500,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

rf.fit(X, y)

importance_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf.feature_importances_
}).sort_values("Importance", ascending=False)

# Evaluate AUC for increasing number of features
feature_counts = list(range(5, min(101, len(X.columns)+1), 5))
auc_scores = []

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

for n in feature_counts:

    top_features = importance_df["Feature"].iloc[:n]

    auc = cross_val_score(
        RandomForestClassifier(
            n_estimators=500,
            random_state=42,
            n_jobs=-1,
            class_weight="balanced"
        ),
        X[top_features],
        y,
        cv=cv,
        scoring="roc_auc",
        n_jobs=-1
    ).mean()

    auc_scores.append(auc)

results_x_rf = pd.DataFrame({
    "n_features": feature_counts,
    "cv_auc": auc_scores
})

print(results_x_rf.sort_values("cv_auc", ascending=False).head())

    n_features    cv_auc
8           45  0.737765
10          55  0.736888
9           50  0.735503
13          70  0.734118
5           30  0.733841


In [28]:
best_n_x = results_x_rf.loc[
    results_x_rf["cv_auc"].idxmax(),
    "n_features"
]

top_features_x_rf = importance_df["Feature"].iloc[:best_n_x].tolist()

top10_features_x_rf = importance_df["Feature"].iloc[:10].tolist()

X_train_rad_x_rf10 = X[top10_features_x_rf].copy()

In [31]:
X_test_rad_x_rf10 = X_test_rad_x[X_train_rad_x_rf10.columns].copy()

In [33]:
from scipy.stats import spearmanr
import pandas as pd

corr_results = []

for col in X_train_clinical_x.columns:
    corr, p = spearmanr(X_train_clinical_x[col], y_train_x)

    corr_results.append({
        "Feature": col,
        "Spearman_r": corr,
        "p_value": p
    })

corr_df = pd.DataFrame(corr_results)
corr_df = corr_df.sort_values(
    by="Spearman_r",
    key=abs,
    ascending=False
)

corr_df

,Feature,Spearman_r,p_value
16,Parotid_R_Dmean,0.219546,0.000001
3,(SEX) Gender,-0.142556,0.001841
5,(SMOKST) Smoking Status,-0.126534,0.005752
6,(ALCOHST) Alcohol Consumption,-0.109173,0.017302
7,(SIGNYN) Exisiting Signs and Symptoms,-0.108788,0.017703
18,Submandibular_gland_R_Dmean,0.099916,0.029456
8,chemo_unfit,-0.087060,0.057955
15,Parotid_L_Dmean,0.081143,0.077275
17,Submandibular_gland_L_Dmean,-0.072430,0.114914
10,Dose_per_fraction,0.067740,0.140437


In [36]:
remove_cols = [
    '(PTUMLOC) Localization of the Primary Tumor',
    'Esophagus_Dmean',
    'Mandible_Dmean',
    'PCM_(constrictors)_Dmean'
]

X_train_clinical_x = X_train_clinical_x.drop(columns=remove_cols, errors='ignore')
X_test_clinical_x  = X_test_clinical_x.drop(columns=remove_cols, errors='ignore')

In [38]:
categorical_cols = [
    '(TNMT) Primary Tumor Clinical T Stage',
    '(TNMN) Primary Tumor Clinical N Stage',
    '(SMOKST) Smoking Status',
    '(ALCOHST) Alcohol Consumption'
]

# Xerostomia
X_train_clinical_x = pd.get_dummies(
    X_train_clinical_x,
    columns=categorical_cols,
    drop_first=True
)

X_test_clinical_x = pd.get_dummies(
    X_test_clinical_x,
    columns=categorical_cols,
    drop_first=True
)

# Align columns
X_train_clinical_x, X_test_clinical_x = X_train_clinical_x.align(
    X_test_clinical_x,
    join='left',
    axis=1,
    fill_value=0
)

In [43]:
# Xerostomia
X_train_combined_x = pd.concat(
    [X_train_rad_x_rf10, X_train_clinical_x],
    axis=1
)

X_test_combined_x = pd.concat(
    [X_test_rad_x_rf10, X_test_clinical_x],
    axis=1
)

print(X_train_combined_x.shape)
print(X_test_combined_x.shape)

(475, 34)
(119, 34)


In [44]:
import os
import pandas as pd

from sklearn.preprocessing import StandardScaler

# Folder
os.chdir(r"C:\Users\moham\Downloads\Projects\Toxicity_Prediction\Drymouth\Datasets")

# Standardize using training data only
scaler = StandardScaler()

X_train_x_scaled = pd.DataFrame(
    scaler.fit_transform(X_train_combined_x),
    columns=X_train_combined_x.columns,
    index=X_train_combined_x.index
)

X_test_x_scaled = pd.DataFrame(
    scaler.transform(X_test_combined_x),
    columns=X_test_combined_x.columns,
    index=X_test_combined_x.index
)

# Save exactly with your names
X_train_x_scaled.to_excel("X_train_x.xlsx", index=True)
X_test_x_scaled.to_excel("X_test_x.xlsx", index=True)

y_train_x.to_frame(name="Drymouth_binary").to_excel(
    "y_train_x.xlsx",
    index=True
)

y_test_x.to_frame(name="Drymouth_binary").to_excel(
    "y_test_x.xlsx",
    index=True
)

print("Files saved successfully.")

Files saved successfully.
